# Simple starter notebook using SLURM API

In [1]:
import os
import requests
import json
import time
from pathlib import Path
import s3fs

In [2]:
with open(os.path.expanduser("~/.tokens/slurm"), "r") as f:
    SLURM_JWT = f.read().strip().split("=")[1]
os.environ["SLURM_JWT"] = SLURM_JWT

In [3]:
os.environ["FI_CXI_RX_MATCH_MODE"] = "hybrid"
os.environ["SBATCH_EXPORT"] = "FI_CXI_RX_MATCH_MODE,SBATCH_EXPORT"

In [4]:
SLURM_addr = "http://172.24.38.181/slurm/v0.0.40"
SLURMdb_addr = "http://172.24.38.181/slurmdb/v0.0.40"
cirrus_username = "dmlsstdev"

In [5]:
headers = {
    "X-SLURM-USER-TOKEN": SLURM_JWT,
    "X-SLURM-USER-NAME": cirrus_username
}
response = requests.get(f"{SLURM_addr}/diag", headers=headers)
diag_info = response.json()
# print(diag_info['statistics']['rpcs_by_user'])
print([d for d in diag_info['statistics']['rpcs_by_user'] if d['user'] == cirrus_username])

[{'user': 'dmlsstdev', 'user_id': 26667, 'count': 437, 'average_time': 1672, 'total_time': 730846}]


In [6]:
response = requests.get(f"{SLURM_addr}/partitions", headers=headers)
partitions = response.json()
partition_name = [ partition['name'] for partition in partitions['partitions'] ][0]
assert partition_name is not None
assert partition_name == "standard"

In [8]:
response = requests.get(f"{SLURM_addr}/partition/{partition_name}", headers=headers)
standard_partition = response.json()
standard_partition['partitions'][0]['qos']['allowed']
assert 'standard' in standard_partition['partitions'][0]['qos']['allowed']

In [9]:
def submit_job(script=None, workdir="/work/dc164/dc164/dmlsstdev/runs", name='', account=None, partition='standard', qos='standard', tasks=1, cpus_per_task=1, time_limit_number=10, comment="Testing SLURM API"):
    assert script is not None and account is not None, "Script and account must be provided to submit a job"
    job_desc = {
        "name": name,
        "account": account,
        "comment": comment,
        "partition": partition,
        "qos": qos,
        "tasks": tasks,
        "cpus_per_task": cpus_per_task,
        "time_limit": {
            "set": True,
            "infinite": False,
            "number": time_limit_number,
        },
        "contiguous": False, # required despite being default
        # "user": "dmlsstdev",
        "current_working_directory": workdir, # required
        "environment": [ # required
            "FI_CXI_RX_MATCH_MODE=hybrid", # required
            "SBATCH_EXPORT=FI_CXI_RX_MATCH_MODE,SBATCH_EXPORT",
        ],
    }
    job = {
        "script": script,
        "job": job_desc,
    }
    response = requests.post(f"{SLURM_addr}/job/submit", headers=headers, json=job)
    return response.json().get("job_id", None)

### Job 1 - run a Dask cluster for 5 minutes

In [10]:
walltime_hours = 0
walltime_minutes = 5
script = f"""#!/bin/bash --login
date
source /etc/bashrc
export WORK=/work/dc164/dc164/${USER}
source $WORK/../shared/lsst/lsst_stack/w_latest/loadLSST.bash
setup lsst_distrib
py_script=$WORK/tarcs/somerville-integration/cirrus-side/dask_cluster.py
python $py_script --account dc164 \
    --queue standard \
    --num-cpus-per-node 280 \
    --threads-per-worker 2 \
    --memory 720GB \
    --walltime {str(walltime_hours).zfill(2)}:{str(walltime_minutes).zfill(2)}:00 \
    --num-nodes 1 \
    --s3-uri s3://tarcs/dask_scheduler_info.txt \
    --s3-endpoint-url https://somerville.ed.ac.uk:6780
date
"""
name = "DaskShd"
account = "dc164"

In [ ]:
dask_clust_jobid = submit_job(script=script, name=name, account=account, time_limit_number=5)

In [15]:
response = requests.get(f"{SLURMdb_addr}/job/{dask_clust_jobid}", headers=headers).json()
print(json.dumps(response, indent=2))

{
  "jobs": [
    {
      "account": "dc164",
      "comment": {
        "administrator": "",
        "job": "",
        "system": ""
      },
      "allocation_nodes": 1,
      "array": {
        "job_id": 0,
        "limits": {
          "max": {
            "running": {
              "tasks": 0
            }
          }
        },
        "task_id": {
          "set": false,
          "infinite": false,
          "number": 0
        },
        "task": ""
      },
      "association": {
        "account": "dc164",
        "cluster": "cirrus",
        "partition": "",
        "user": "dmlsstdev",
        "id": 292
      },
      "block": "",
      "cluster": "cirrus",
      "constraints": "",
      "container": "",
      "derived_exit_code": {
        "status": [
          "SUCCESS"
        ],
        "return_code": {
          "set": true,
          "infinite": false,
          "number": 0
        },
        "signal": {
          "id": {
            "set": false,
            "infinit

In [25]:
response = requests.get(f"{SLURMdb_addr}/jobs", headers=headers)
myjobs = [ job for job in response.json().get("jobs", []) if account in job.get("account", "") ]
del response

In [26]:
myjob_ids = []
for job in myjobs:
    myjob_ids.append(job.get("job_id", "N/A"))
    print(f"Job ID: {job.get('job_id', 'N/A')}, Job Name: {job.get('name', 'N/A')}, Job Account: {job.get('account', 'N/A')}, Job State: {job.get('job_state', ['N/A'])[0]}")

Job ID: 105951, Job Name: run.slurm, Job Account: dc164, Job State: TIMEOUT
Job ID: 109758, Job Name: APITest, Job Account: dc164, Job State: FAILED
Job ID: 109759, Job Name: APITest, Job Account: dc164, Job State: FAILED
Job ID: 110129, Job Name: APITest, Job Account: dc164, Job State: COMPLETED
Job ID: 110142, Job Name: APITest, Job Account: dc164, Job State: COMPLETED
Job ID: 110294, Job Name: APITest, Job Account: dc164, Job State: COMPLETED
Job ID: 110705, Job Name: APITest, Job Account: dc164, Job State: COMPLETED
Job ID: 110712, Job Name: APITest, Job Account: dc164, Job State: COMPLETED
Job ID: 110714, Job Name: APITest, Job Account: dc164, Job State: CANCELLED
Job ID: 110716, Job Name: dask-worker, Job Account: dc164, Job State: COMPLETED
Job ID: 111216, Job Name: APITest, Job Account: dc164, Job State: COMPLETED
Job ID: 111217, Job Name: dask-worker, Job Account: dc164, Job State: CANCELLED
Job ID: 111359, Job Name: LogUploader, Job Account: dc164, Job State: FAILED
Job ID: 1

In [27]:
len(myjobs)

24

In [28]:
myjob_ids

[105951,
 109758,
 109759,
 110129,
 110142,
 110294,
 110705,
 110712,
 110714,
 110716,
 111216,
 111217,
 111359,
 111360,
 111361,
 111362,
 111367,
 111389,
 112035,
 112036,
 112037,
 112039,
 112040,
 112041]

In [29]:
response = requests.get(f"{SLURMDB_addr}/job/{112039}", headers=headers)
job_stat = response.json()
print(json.dumps(job_stat, indent=2))

{
  "jobs": [
    {
      "account": "dc164",
      "comment": {
        "administrator": "",
        "job": "Testing SLURM API",
        "system": ""
      },
      "allocation_nodes": 1,
      "array": {
        "job_id": 0,
        "limits": {
          "max": {
            "running": {
              "tasks": 0
            }
          }
        },
        "task_id": {
          "set": false,
          "infinite": false,
          "number": 0
        },
        "task": ""
      },
      "association": {
        "account": "dc164",
        "cluster": "cirrus",
        "partition": "",
        "user": "dmlsstdev",
        "id": 292
      },
      "block": "",
      "cluster": "cirrus",
      "constraints": "",
      "container": "",
      "derived_exit_code": {
        "status": [
          "SUCCESS"
        ],
        "return_code": {
          "set": true,
          "infinite": false,
          "number": 0
        },
        "signal": {
          "id": {
            "set": false,
   

### Job 2 - upload outputs from Job 1

In [34]:
prev_job_stat.get("jobs", {})[0]

{'account': 'dc164',
 'comment': {'administrator': '', 'job': 'Testing SLURM API', 'system': ''},
 'allocation_nodes': 1,
 'array': {'job_id': 0,
  'limits': {'max': {'running': {'tasks': 0}}},
  'task_id': {'set': False, 'infinite': False, 'number': 0},
  'task': ''},
 'association': {'account': 'dc164',
  'cluster': 'cirrus',
  'partition': '',
  'user': 'dmlsstdev',
  'id': 292},
 'block': '',
 'cluster': 'cirrus',
 'constraints': '',
 'container': '',
 'derived_exit_code': {'status': ['SUCCESS'],
  'return_code': {'set': True, 'infinite': False, 'number': 0},
  'signal': {'id': {'set': False, 'infinite': False, 'number': 0},
   'name': ''}},
 'time': {'elapsed': 543,
  'eligible': 1773769273,
  'end': 1773769829,
  'start': 1773769286,
  'submission': 1773769273,
  'suspended': 0,
  'system': {'seconds': 0, 'microseconds': 0},
  'limit': {'set': True, 'infinite': False, 'number': 10},
  'total': {'seconds': 0, 'microseconds': 0},
  'user': {'seconds': 0, 'microseconds': 0}},
 'exit

In [35]:
prev_jobid = 112039
response = requests.get(f"{SLURMdb_addr}/job/{prev_jobid}", headers=headers)
prev_job_stat = response.json()
prev_workdir = prev_job_stat.get("jobs", {})[0].get("working_directory", ["N/A"])
print(f"Previous job working directory: {prev_workdir}")
script = f"""#!/bin/bash --login
date
source /etc/bashrc
export WORK=/work/dc164/dc164/${{USER}}
source $WORK/../shared/lsst/lsst_stack/w_latest/loadLSST.bash
setup lsst_distrib
py_script=$WORK/tarcs/somerville-integration/cirrus-side/log_uploader.py
python $py_script --s3-bucket tarcs \
    --jobid {prev_jobid} \
    --workdir {prev_workdir} \
    --creds-file /work/dc164/dc164/dmlsstdev/.aws/credentials \
"""

Previous job working directory: /work/dc164/dc164/dmlsstdev/runs


In [36]:
upload_jobid = submit_job(script=script, name="LogUploader", account=account, comment="Uploading logs from previous job")

In [38]:
upload_job_stats = requests.get(f"{SLURMdb_addr}/job/{upload_jobid}", headers=headers).json()
print(upload_job_stats['jobs'][0].get('state'))

{'current': ['COMPLETED'], 'reason': 'None'}


### Fetch logs from s3

In [39]:
with open(os.path.expanduser('~/.aws/credentials'), 'r') as credf:
    creds = {
        l.split(' = ')[0].strip():l.split(' = ')[1].strip() for l in credf.readlines() if l.startswith('aws')
    }
tarcs_s3 = s3fs.S3FileSystem(
      key=creds['aws_access_key_id'],
      secret=creds['aws_secret_access_key'],
      endpoint_url='https://somerville.ed.ac.uk:6780'
   )


In [47]:
tarcs_s3.ls('tarcs/jobid')

['tarcs/jobid/111216_logs.tar.gz', 'tarcs/jobid/112039_logs.tar.gz']

In [48]:
cwd = Path(os.getcwd())

In [51]:
cwd / '112039_logs.tar.gz'

PosixPath('/home/davedavemckay/tarcs/somerville-integration/sville-side/112039_logs.tar.gz')

In [55]:
tarcs_s3.get('tarcs/jobid/112039_logs.tar.gz', str(cwd / '112039_logs.tar.gz'))

[None]

In [66]:
os.listdir(cwd)

['TARCS_controller.ipynb',
 's3fs_test.ipynb',
 'rubin_dp1.ipynb',
 '112039_logs.tar.gz',
 'slurm-112039.out',
 '__pycache__',
 '.ipynb_checkpoints',
 '102_5_LSDB_data_access.ipynb',
 'SLURMAPIConnection.py',
 'SLURMAPI_test.ipynb']

In [57]:
import tarfile

In [71]:
with tarfile.open('112039_logs.tar.gz', 'r:gz') as targz:
    logs = targz.extractall()
    # for log in logs:
    #     targz.extractfile(log)

/tmp/ipykernel_553/364613735.py:2: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  logs = targz.extractall()


In [72]:
!cat 'slurm-112039.out'

/var/spool/slurm/d/job112039/slurm_script: line 2: datedd: command not found
Creating user data directory: /home/dc164/dc164/dmlsstdev/.eups
setup: [Errno 13] Permission denied: '/home/dc164'
/mnt/lustre/e1000/home/dc164/dc164/shared/lsst/lsst_stack/w_2026_11/conda/envs/lsst-scipipe-12.1.0/etc/conda/activate.d/eups_activate.sh: line 45: export: setup: not a function
/mnt/lustre/e1000/home/dc164/dc164/shared/lsst/lsst_stack/w_2026_11/conda/envs/lsst-scipipe-12.1.0/etc/conda/activate.d/eups_activate.sh: line 47: export: unsetup: not a function
/var/spool/slurm/d/job112039/slurm_script: line 6: setup: command not found
/mnt/lustre/e1000/home/dc164/dc164/shared/lsst/lsst_stack/w_2026_11/conda/envs/lsst-scipipe-12.1.0/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41423 instead
  warnings.warn(
Dashboard link: http://10.62.0.57:41423/status
Scheduler address: tcp://10